# 核函数定义与 Kernel 参数

Kernel 是在 AI Core 上执行的函数，在函数签名中定义输入、输出和运行时参数，在函数体中组织数据搬运与计算。本节讲清 Kernel 的声明方式、参数体系与启动配置。


## 环境准备

本节代码面向 Ascend 950 环境。运行前先完成基础环境配置（详见 01.06）：设置设备号、关闭设备后端自动加载，并完成导入自检。

In [ ]:
import os

os.environ.setdefault('TILE_FWK_DEVICE_ID', '0')
os.environ['TORCH_DEVICE_BACKEND_AUTOLOAD'] = '0'

import pypto_pro.language as pl
import torch
import torch_npu

print('导入完成；设备编译、执行与精度仍需在 950 验证')

## 1. 使用 JIT 装饰器声明 Kernel

Kernel 函数必须使用 `@pypto_pro.language.jit()` 装饰。下表列出本课程使用的主要参数；示意函数省略计算体：

```python
import pypto_pro.language as pl


@pl.jit(auto_mutex=True, compile_timeout=1200, name="add_kernel")
def add_kernel(
    x: pl.Tensor[[8, 256], pl.DT_FP32],
    y: pl.Tensor[[8, 256], pl.DT_FP32],
    out: pl.Tensor[[8, 256], pl.DT_FP32],
):
    ...
```

<table style="margin-left:0; margin-right:auto; text-align:left;">
<tr><th style="text-align:left;">参数</th><th style="text-align:left;">说明</th><th style="text-align:left;">默认值</th></tr>
<tr><td>arch</td><td>编译目标；通常省略，由运行环境自动确定</td><td>None</td></tr>
<tr><td>auto_mutex</td><td>根据 TileGroup 的 mutex 元数据，为框架能识别的数据依赖自动插入同步</td><td>True</td></tr>
<tr><td>name</td><td>自定义 Kernel 名称，用于区分编译产物</td><td>函数名</td></tr>
<tr><td>pipeline</td><td>配置自动 CV 并行流水变换（第 8 章展开）</td><td>None</td></tr>
<tr><td>sanitizer</td><td>开启 Kernel 内存检测（越界/重叠/mutex 配对，第 8 章展开）</td><td>False</td></tr>
<tr><td>tiling_key</td><td>绑定 TilingKey 字段（第 4 章展开）</td><td>None</td></tr>
<tr><td>datatype</td><td>声明需要数据类型特化的参数</td><td>None</td></tr>
<tr><td>compile_timeout</td><td>编译超时（秒）</td><td>600</td></tr>
</table>

两条纪律贯穿本章：**Kernel 不返回 Python 值**——计算结果只能写回 Tensor/Ptr 对应的 GM 区域；**Kernel 必须用 `@pl.jit` 装饰**——普通 Python 函数不会被编译，调用时直接报错。


## 2. 四种参数类型

<table style="margin-left:0; margin-right:auto; text-align:left;">
<tr><th style="text-align:left;">参数类型</th><th style="text-align:left;">类型标注示例</th><th style="text-align:left;">用途</th></tr>
<tr><td><b>Tensor</b></td><td><code>x: pl.Tensor[[64, 64], pl.DT_FP16]</code></td><td>接收 GM 多维数据；声明 shape、dtype 与可选 layout</td></tr>
<tr><td><b>Ptr</b></td><td><code>x: pl.Ptr[pl.DT_FP16]</code></td><td>接收裸指针；通常用 make_tensor 构造 Tensor 视图（配 TilingData 提供动态 shape）</td></tr>
<tr><td><b>运行时标量</b></td><td><code>alpha: pl.DT_FP32</code></td><td>接收整型/浮点标量，值在运行时传入</td></tr>
<tr><td><b>TilingData</b></td><td><code>tiling: OpTiling</code>（dataclass）</td><td>接收 shape、stride、循环边界等结构化运行时参数（第 4 章展开）</td></tr>
</table>

Tensor 参数适合直接使用调用侧已知 shape；Ptr + TilingData 组合适合动态 shape 场景。


## 3. Tensor 的 Shape 声明

Tensor 每一维有四种声明方式，直接影响编译变体的数量（详见 02.07）：

<table style="margin-left:0; margin-right:auto; text-align:left;">
<tr><th style="text-align:left;">声明方式</th><th style="text-align:left;">含义</th><th style="text-align:left;">维度变化时</th></tr>
<tr><td>正整数</td><td>固定维度，启动时必须一致</td><td>尺寸固定</td></tr>
<tr><td><code>pl.DYNAMIC</code></td><td>运行时动态维度</td><td>复用同一编译变体</td></tr>
<tr><td><code>pl.STATIC</code></td><td>编译期特化维度</td><td>生成新编译变体</td></tr>
<tr><td>末尾 <code>...</code></td><td>展开剩余维度，各维按 STATIC</td><td>rank 或维度变化生成新变体</td></tr>
</table>

```python
# 固定、动态、编译期特化维度混合声明
x: pl.Tensor[[64, pl.DYNAMIC, pl.STATIC], pl.DT_FP16]

# rank 运行时确定；首维 DYNAMIC，其余按 STATIC
y: pl.Tensor[[pl.DYNAMIC, ...], pl.DT_FP16]
```

【待 950 验证：以上为官方文档示例】


## 4. 启动 Kernel 与核数配置

Kernel 通过下标形式启动：`kernel[stream, block_dim](...)`。`stream` 为 None 时使用当前流；`block_dim` 配置核数，纯 Vector Kernel 的上限可查询获得：

```python
import torch
import torch_npu

stream = torch.npu.current_stream()
limits = torch.npu.get_stream_limit(stream)
max_vector_blocks = limits['vector_core_num']

add_kernel[stream, 1](x, y, out)   # 01.06 的单 Tile 示例没有多核切分，只启动 1 核
stream.synchronize()                # Kernel 下发是异步的，读取结果前先同步
```

【待 950 验证：get_stream_limit 用法取自官方 JIT 文档】

单 Tile Kernel 若增加核数却不按核索引划分工作，多个核会写同一输出区域。为避免歧义，本课程对未做切分的示例统一显式指定 1 核；不加方括号的普通函数式调用行为以发行包为准。

核数不是越多越好：核数应与数据分片数量匹配（第 4 章多核切分展开）。


## 5. 完整示例走读：带标量的缩放算子

把本节知识串成一个 Kernel 定义示例——`out = alpha * x + beta`（[1, 64] FP32）：

```python
@pl.jit(auto_mutex=True)
def scale_shift_kernel(
    x: pl.Tensor[[1, 64], pl.DT_FP32],
    alpha: pl.DT_FP32,
    beta: pl.DT_FP32,
    out: pl.Tensor[[1, 64], pl.DT_FP32],
):
    tt = pl.TileType(shape=[1, 64], dtype=pl.DT_FP32, target_memory=pl.MemorySpace.Vec)
    x_group = pl.make_tile_group(type=tt, addrs=0x0000, mutex_ids=[0])
    out_group = pl.make_tile_group(type=tt, addrs=0x0100, mutex_ids=[1])

    with pl.section_vector():
        x_tile = x_group.current()
        out_tile = out_group.current()
        pl.load(x_tile, x, [0, 0])
        pl.mul(out_tile, x_tile, alpha)       # Tile × Scalar
        pl.add(out_tile, out_tile, beta)      # 原地累加（Tile add API 明确支持）
        pl.store(out, out_tile, [0, 0])
```

【待 950 验证：完整示例（kernel_function.md 标量参数 + tile 计算模式组合推导）】

**逐段解读**（对照本节知识）：

1. **签名层**：Tensor 参数带静态 shape `[1, 64]`（编译期固定）；`alpha/beta` 是 `pl.DT_FP32` 运行时标量——值变化**不触发**重编译（02.07 展开），Host 侧每次调用可传不同值；
2. **Tile 层**：两块 Tile 各 64×4B = 256B = 0x100，地址 0x0000 与 0x0100 恰好不重叠（且满足 32B 对齐）——**先算字节数再定地址**的习惯从这里养成；
3. **计算层**：`mul` 的第三参数是标量（Tile-Scalar 模式，第 3 章）；`add(out, out, beta)` 是原地写法——**原地支持与否以 API 文档为准**，Tile `add.md` 已明确支持 out 与 lhs 为同一 Tile；实际执行仍待 950 验证。

**常见陷阱**：

- 把 Tensor 支持的 dtype 一律当成 Host 标量 ABI 支持的 dtype；当前运行时浮点标量入口使用 FP32，本例保留 `pl.DT_FP32`；
- 忘写 `pl.jit` → 函数是普通 Python 函数，调用报错；
- Tensor 静态 shape 与实参 shape 不一致 → 启动时报维度不匹配。

**思考题**：把这个 Kernel 的 x/out 改成 `[pl.DYNAMIC, 64]`，并用 `x.shape[0]` 驱动逐行循环、逐行 load/store（Tile 仍为 `[1,64]`），调用时传 [1,64] 和 [16,64] 两种 shape——编译发生几次？仅修改标注不会自动覆盖新增的行。（答案见 02.07；先用你的直觉猜一下。）


## 6. Kernel 内感知多核环境

Kernel 内可用两个接口感知多核环境：

```python
num_cores = pl.get_block_num()   # 实际生效核数
core_id = pl.get_block_idx()     # 当前核索引
```

配合 `pl.range(core_id, tile_num, num_cores)` 即可实现"第 core_id 个核以 num_cores 为步长认领数据块"的 SPMD 分片。


### 规格检查：只把 shape 标成 DYNAMIC 还缺什么？

固定 `[8,256]` 的 Kernel 声明了精确契约。传入 `[9,256]` 不能理解为自动重编译后就正确；固定维不匹配会被拒绝。若把第一维标成 `pl.DYNAMIC`，代码还必须从 `x.shape[0]` 取得行数，并遍历全部行或全部行 Tile。

以每次 4 行为例，9 行需要 3 个任务，起点 0、4、8，有效行数分别为 4、4、1。只改参数标注、仍只搬入 `[0,0]` 的 4 行，会留下 5 行未计算。动态尺寸解决“参数如何表示”，循环和有效形状解决“数据如何覆盖”。Host 还应保证两个输入 shape 相同，输出大小匹配。


## 7. 课后练习

1. （填空）声明"运行时才确定、且不希望触发重新编译"的维度，应使用 `pl.Tensor` 的 ______ 声明方式。
2. （选择）`stream` 和 `block_dim` 的调整会影响编译实例吗？
   A. 会，两者都参与编译签名<br/>
   B. 不会，两者只影响本次启动
3. （实践）把第 1 章向量加法 Kernel 的首维改为 DYNAMIC，并改为逐行 Tile 加法，循环范围取 `x.shape[0]`。思考：调用时传入 [16, 256] 与 [32, 256] 的 Tensor，会触发几次编译？

In [ ]:
!cat ./answer/02.02_answer.txt

## 8. 本节小结

- Kernel = `@pl.jit(...)` 装饰的函数 + 类型标注的参数（Tensor/Ptr/标量/TilingData）；
- Tensor 维度声明四方式（固定/DYNAMIC/STATIC/...）决定编译变体数量；
- 启动形式 `kernel[stream, block_dim](...)`；`get_stream_limit` 查核数上限；Kernel 内 `get_block_num()/get_block_idx()` 感知多核。

下一节展开三类数据对象的细节。